# HbA1c — In-sample XAI extension

The reusable XAI framework is applied to the final logistic-regression interaction model fitted on the eligible full cohort.

The same cohort supplies both reference and explanation data. All explanations and ROC-AUC scoring are therefore in-sample and do not estimate out-of-sample predictive performance.

Repeated-fit stability is not assessed because the implementation provides no retraining callback. Results are exported to `../../results/hba1c`.

## Minimal cohort and final-model preparation

In [1]:
import pandas as pd
import numpy as np
import re
import statsmodels.formula.api as smf


data_path = "../../data/hba1c/diabetic_data_initial.csv"
map_path = "../../data/hba1c/id_mapping.csv"

df = pd.read_csv(data_path, keep_default_na=False)
idmap_raw = pd.read_csv(map_path, keep_default_na=False)



In [2]:
# -----------------------------
# Parse ID mapping file
# -----------------------------
idmap_raw.columns = ["id", "description"]
idmap_raw["id"] = idmap_raw["id"].astype(str).str.strip()
idmap_raw["description"] = idmap_raw["description"].astype(str).str.strip()


In [3]:
def parse_id_mapping(idmap_raw):
    sections = {}
    current_section = None
    current_rows = []

    for _, row in idmap_raw.iterrows():
        left = str(row["id"]).strip()
        right = str(row["description"]).strip()

        if left in ["nan", "ID mappings for the diabetes dataset"]:
            continue
        if right in ["nan", "? - always means a missing value"]:
            continue

        if left in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
            if current_section is not None and current_rows:
                sections[current_section] = pd.DataFrame(current_rows, columns=["id", "description"])
            current_section = left
            current_rows = []
            continue

        if left == "description" or right == "description":
            continue

        try:
            key = int(float(left))
            current_rows.append((key, right))
        except Exception:
            pass

    if current_section is not None and current_rows:
        sections[current_section] = pd.DataFrame(current_rows, columns=["id", "description"])

    return sections

sections = parse_id_mapping(idmap_raw)

In [4]:
admission_type_map = dict(zip(sections["admission_type_id"]["id"], sections["admission_type_id"]["description"]))
discharge_map = dict(zip(sections["discharge_disposition_id"]["id"], sections["discharge_disposition_id"]["description"]))
admission_source_map = dict(zip(sections["admission_source_id"]["id"], sections["admission_source_id"]["description"]))

In [5]:
# -----------------------------
# Main preprocessing
# -----------------------------
df2 = df.copy()

# outcome -> readmitted within 30 days
df2["readmitted_bin"] = (df2["readmitted"] == "<30").astype(int)

# dropping variables excluded in the paper
df2 = df2.drop(columns=["weight", "payer_code"], errors="ignore")

# replacing missing markers
df2["medical_specialty"] = df2["medical_specialty"].replace("?", np.nan).fillna("missing")
df2["race"] = df2["race"].replace("?", np.nan).fillna("Missing")

# adding mapped descriptions
df2["discharge_desc"] = df2["discharge_disposition_id"].map(discharge_map)
df2["admission_type_desc"] = df2["admission_type_id"].map(admission_type_map)
df2["admission_source_desc"] = df2["admission_source_id"].map(admission_source_map)

# keeping only the first encounter per patient
# using drop_duplicates, not groupby.first()
df2 = (
    df2.sort_values(["patient_nbr", "encounter_id"])
       .drop_duplicates(subset="patient_nbr", keep="first")
       .copy()
)

# excluding invalid/unknown gender
df2 = df2[df2["gender"].isin(["Female", "Male"])].copy()

# removing discharge dispositions related to death/hospice
death_hospice_ids = [11, 13, 14, 19, 20, 21]
df2 = df2[~df2["discharge_disposition_id"].isin(death_hospice_ids)].copy()

print("Shape after cohort selection:", df2.shape)

Shape after cohort selection: (69970, 52)


In [6]:
# -----------------------------
# Derived variables
# -----------------------------
def make_hba1c_group(row):
    a1c = str(row["A1Cresult"]).strip()
    change = str(row["change"]).strip()

    if a1c == "None":
        return "No test was performed"
    elif a1c in ["Norm", ">7"]:
        return "Normal result of the test"
    elif a1c == ">8" and change == "No":
        return "Result was high but the diabetic medication was not changed"
    elif a1c == ">8" and change == "Ch":
        return "Result was high and the diabetic medication was changed"
    else:
        return np.nan



df2["hba1c_group"] = df2.apply(make_hba1c_group, axis=1)

In [7]:
df2["discharge_group"] = np.where(
    df2["discharge_disposition_id"] == 1,
    "Discharged to home",
    "Otherwise"
)

In [8]:
def map_admission_source(x):
    if x == 7:
        return "Admitted from emergency room"
    elif x in [1, 2, 3]:
        return "Admitted because of physician/clinic referral"
    else:
        return "Otherwise"

df2["admission_source_group"] = df2["admission_source_id"].apply(map_admission_source)


In [9]:
def map_specialty(x):
    if x == "InternalMedicine":
        return "Internal Medicine"
    elif x == "Cardiology":
        return "Cardiology"
    elif "Surgery" in str(x):
        return "Surgery"
    elif x in ["Family/GeneralPractice", "FamilyPractice"]:
        return "Family/general practice"
    elif x == "missing":
        return "Missing or unknown"
    else:
        return "Other"

df2["specialty_group"] = df2["medical_specialty"].apply(map_specialty)

In [10]:
def map_race(x):
    if x == "AfricanAmerican":
        return "African American"
    elif x == "Caucasian":
        return "Caucasian"
    elif x == "Missing":
        return "Missing"
    else:
        return "Other"

df2["race_group"] = df2["race"].apply(map_race)


In [11]:
def age_midpoint(age_str):
    nums = [int(x) for x in str(age_str).replace("[", "").replace(")", "").split("-") if x.isdigit()]
    if len(nums) == 2:
        return (nums[0] + nums[1]) / 2
    return np.nan

df2["age_mid"] = df2["age"].apply(age_midpoint)

def map_age_group3(x):
    if pd.isna(x):
        return np.nan
    elif x <= 30:
        return "30 years old or younger"
    elif x <= 60:
        return "30-60 years old"
    else:
        return "Older than 60"

df2["age_group3"] = df2["age_mid"].apply(map_age_group3)

In [12]:
def group_primary_diag(code):
    code = str(code).strip()

    if code.startswith("V") or code.startswith("E"):
        return "Other"
    if code.startswith("250"):
        return "Diabetes"

    m = re.match(r"^(\d+)", code)
    if not m:
        return "Other"

    num = int(m.group(1))

    if 390 <= num <= 459 or num == 785:
        return "Circulatory"
    elif 460 <= num <= 519 or num == 786:
        return "Respiratory"
    elif 520 <= num <= 579 or num == 787:
        return "Digestive"
    elif 800 <= num <= 999:
        return "Injury"
    elif 710 <= num <= 739:
        return "Musculoskeletal"
    elif 580 <= num <= 629 or num == 788:
        return "Genitourinary"
    elif 140 <= num <= 239:
        return "Neoplasms"
    else:
        return "Other"

df2["primary_diag_group"] = df2["diag_1"].apply(group_primary_diag)

In [13]:
# -----------------------------
# Final analysis dataset
# -----------------------------
df2["hba1c_group"] = df2.apply(make_hba1c_group, axis=1)

analysis_df = df2[[
    "readmitted_bin",
    "hba1c_group",
    "gender",
    "discharge_group",
    "admission_source_group",
    "specialty_group",
    "primary_diag_group",
    "race_group",
    "age_group3",
    "age_mid",
    "time_in_hospital"
]].dropna().copy()

print("analysis_df shape:", analysis_df.shape)
print(analysis_df["hba1c_group"].value_counts(dropna=False))


analysis_df shape: (69970, 11)
hba1c_group
No test was performed                                          57125
Normal result of the test                                       6606
Result was high and the diabetic medication was changed         4058
Result was high but the diabetic medication was not changed     2181
Name: count, dtype: int64


In [14]:
analysis_df["discharge_group"] = pd.Categorical(
    analysis_df["discharge_group"],
    categories=["Discharged to home", "Otherwise"]
)

analysis_df["race_group"] = pd.Categorical(
    analysis_df["race_group"],
    categories=["African American", "Caucasian", "Missing", "Other"]
)

analysis_df["admission_source_group"] = pd.Categorical(
    analysis_df["admission_source_group"],
    categories=[
        "Admitted from emergency room",
        "Admitted because of physician/clinic referral",
        "Otherwise"
    ]
)

analysis_df["specialty_group"] = pd.Categorical(
    analysis_df["specialty_group"],
    categories=[
        "Cardiology",
        "Family/general practice",
        "Internal Medicine",
        "Missing or unknown",
        "Other",
        "Surgery"
    ]
)

analysis_df["age_group3"] = pd.Categorical(
    analysis_df["age_group3"],
    categories=[
        "30-60 years old",
        "Older than 60",
        "30 years old or younger"
    ]
)

analysis_df["primary_diag_group"] = pd.Categorical(
    analysis_df["primary_diag_group"],
    categories=[
        "Diabetes",
        "Circulatory",
        "Digestive",
        "Genitourinary",
        "Injury",
        "Musculoskeletal",
        "Neoplasms",
        "Other",
        "Respiratory"
    ]
)



analysis_df["hba1c_group"] = pd.Categorical(
    analysis_df["hba1c_group"],
    categories=[
        "No test was performed",
        "Normal result of the test",
        "Result was high and the diabetic medication was changed",
        "Result was high but the diabetic medication was not changed"
    ],
    ordered=True
)

In [15]:
# making categorical
cat_cols = [
    "hba1c_group", "gender", "discharge_group", "admission_source_group",
    "specialty_group", "primary_diag_group", "race_group", "age_group3"
]

for c in cat_cols:
    analysis_df[c] = analysis_df[c].astype("category")

In [16]:
final_model = smf.logit(
    "readmitted_bin ~ gender + race_group + admission_source_group + "
    "discharge_group + specialty_group + primary_diag_group + "
    "age_group3 + time_in_hospital + hba1c_group + "
    "discharge_group:race_group + "
    "discharge_group:specialty_group + "
    "discharge_group:primary_diag_group + "
    "discharge_group:time_in_hospital + "
    "specialty_group:time_in_hospital + "
    "specialty_group:age_group3 + "
    "primary_diag_group:time_in_hospital + "
    "primary_diag_group:hba1c_group",
    data=analysis_df
).fit(maxiter=200)


Optimization terminated successfully.
         Current function value: 0.294692
         Iterations 8


In [17]:
# ==========================================
# BUILD XAI INPUTS FROM FITTED STATSMODEL
# ==========================================

X_design = pd.DataFrame(
    final_model.model.exog,
    columns=final_model.model.exog_names
)

y_design = pd.Series(
    final_model.model.endog,
    name="readmitted_bin"
)

feature_names = list(X_design.columns)

print("X_design shape:", X_design.shape)
print("y_design shape:", y_design.shape)
print("Number of features:", len(feature_names))

X_design shape: (69970, 91)
y_design shape: (69970,)
Number of features: 91


## Call the existing reusable XAI framework

In [18]:
import sys
from pathlib import Path

FRAMEWORK_DIR = Path.cwd().parent / "xai_framework"

if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from run_pipeline import run_pipeline

c:\Users\catia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
# The framework names these fields X_train/X_test, but this paper has no
# held-out prediction design. Both refer to the eligible full-cohort design
# matrix, and all resulting explanations are labelled in-sample.
config = {
    "dataset_name": "hba1c",

    "model": final_model,

    "X_train": X_design,
    "X_test": X_design,

    "y_train": y_design,
    "y_test": y_design,

    "feature_names": feature_names,

    "scoring": "roc_auc",

    "output_dir": "../../results/hba1c"
}

experiment = run_pipeline(config)

PermutationExplainer explainer: 69971it [42:05, 27.69it/s]                             


Results exported to: ../../results/hba1c


## XAI framework output sanity checks — in-sample

In [20]:
import numpy as np
from pathlib import Path

from core.statsmodels_wrapper import StatsModelsWrapper

In [21]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))

Dataset: hba1c
Model: BinaryResultsWrapper
X_train shape: (69970, 91)
X_test shape: (69970, 91)
y_train length: 69970
y_test length: 69970
Feature names: 91


#### Consistency

In [22]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("\nBasic shape validation: OK")


Basic shape validation: OK


#### HBA1C-SPECIFIC DESIGN MATRIX CHECK

In [23]:
assert list(config["X_train"].columns) == config["feature_names"]
assert list(config["X_test"].columns) == config["feature_names"]

assert list(config["X_train"].columns) == list(final_model.model.exog_names)

print("Statsmodels design-matrix alignment: OK")

Statsmodels design-matrix alignment: OK


#### Fitted model predicts on the configured data

In [24]:
model = config["model"]

wrapped_model = StatsModelsWrapper(model)

probs = wrapped_model.predict_proba(
    config["X_test"]
)[:, 1]

preds = wrapped_model.predict(
    config["X_test"]
)

print("\nProbability range:", probs.min(), probs.max())
print(
    "Predictions:",
    np.unique(preds, return_counts=True)
)

assert np.all(probs >= 0)
assert np.all(probs <= 1)

print("Prediction test: OK")


Probability range: 0.009531647448095457 0.35282606409965933
Predictions: (array([0]), array([69970]))
Prediction test: OK


#### Results from Framework

In [25]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print("\nGlobal result keys:")
print(global_results.keys())

print("\nLocal result keys:")
print(local_results.keys())

if stability_results is None:
    print("\nStability: correctly absent for HbA1c")
else:
    print("\nStability keys:")
    print(stability_results.keys())


Global result keys:
dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'rank_compare', 'rank_agreement'])

Local result keys:
dict_keys(['cases', 'lime'])

Stability: correctly absent for HbA1c


#### Global XAI

In [26]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("\nNative importance:")
display(native_df.head(10))

print("\nPermutation importance:")
display(perm_df.head(10))

print("\nSHAP importance:")
display(shap_df.head(10))



Native importance:


,Feature,Importance,RawValue,Method
50,specialty_group[T.Surgery]:age_group3[T.30 yea...,2.989244,-2.989244,statsmodels
46,specialty_group[T.Family/general practice]:age...,2.498977,-2.498977,statsmodels
49,specialty_group[T.Other]:age_group3[T.30 years...,2.409895,-2.409895,statsmodels
47,specialty_group[T.Internal Medicine]:age_group...,2.023971,-2.023971,statsmodels
21,age_group3[T.30 years old or younger],1.857647,1.857647,statsmodels
48,specialty_group[T.Missing or unknown]:age_grou...,1.471311,-1.471311,statsmodels
71,primary_diag_group[T.Musculoskeletal]:hba1c_gr...,0.802534,0.802534,statsmodels
63,primary_diag_group[T.Musculoskeletal]:hba1c_gr...,0.796584,0.796584,statsmodels
16,primary_diag_group[T.Musculoskeletal],0.726567,-0.726567,statsmodels
72,primary_diag_group[T.Neoplasms]:hba1c_group[T....,0.722161,0.722161,statsmodels



Permutation importance:


,Feature,Importance,Std
76,time_in_hospital,0.065603,0.002451
10,specialty_group[T.Missing or unknown],0.022608,0.001586
7,discharge_group[T.Otherwise],0.015138,0.001527
9,specialty_group[T.Internal Medicine],0.012162,0.001061
80,specialty_group[T.Missing or unknown]:time_in_...,0.011473,0.001050
17,primary_diag_group[T.Musculoskeletal],0.010799,0.001289
21,age_group3[T.Older than 60],0.010546,0.001011
50,specialty_group[T.Other]:age_group3[T.30 years...,0.009578,0.000910
22,age_group3[T.30 years old or younger],0.008207,0.001121
89,primary_diag_group[T.Other]:time_in_hospital,0.007794,0.000958



SHAP importance:


,Feature,MeanAbsSHAP,MeanSHAP
0,time_in_hospital,0.025380,0.000857
1,specialty_group[T.Missing or unknown],0.017390,-0.001094
2,discharge_group[T.Otherwise],0.011746,-0.000497
3,specialty_group[T.Missing or unknown]:time_in_...,0.011398,-0.000303
4,age_group3[T.Older than 60],0.010153,0.001985
5,specialty_group[T.Internal Medicine],0.007391,0.002033
6,primary_diag_group[T.Other]:time_in_hospital,0.007338,0.001248
7,specialty_group[T.Other],0.007266,-0.000364
8,discharge_group[T.Otherwise]:specialty_group[T...,0.007005,-0.000386
9,discharge_group[T.Otherwise]:time_in_hospital,0.006646,0.000541


#### Feature coverage

In [27]:
expected_features = len(config["feature_names"])

if native_df is not None:
    print("\nNative features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", expected_features)


expected_native_features = len([
    f for f in config["feature_names"]
    if str(f).lower() not in ["intercept", "const"]
])

if native_df is not None:
    assert len(native_df) == expected_native_features

print("Native features:", len(native_df))
print("Expected native features:", expected_native_features)


Native features: 90
Permutation features: 91
SHAP features: 91
Expected features: 91
Native features: 90
Expected native features: 90


#### Broken values

In [28]:
print("\nPermutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())

assert perm_df.isna().sum().sum() == 0
assert shap_df.isna().sum().sum() == 0

if native_df is not None:
    assert native_df.isna().sum().sum() == 0

print("NaN validation: OK")


Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0
NaN validation: OK


#### Rank agreement

In [29]:
print("\nRank agreement:")
print(global_results["rank_agreement"])

print("\nRank comparison:")
display(global_results["rank_compare"].head(15))



Rank agreement:
{'Native_vs_SHAP': np.float64(0.045627032143886075), 'Native_vs_Permutation': np.float64(0.40850310737951184), 'SHAP_vs_Permutation': np.float64(0.8828497345351277)}

Rank comparison:


,Feature,Importance_x,RawValue,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,specialty_group[T.Surgery]:age_group3[T.30 yea...,2.989244,-2.989244,statsmodels,1.0,0.000123,-0.000123,75.0,0.000674,0.000205,51.0
1,specialty_group[T.Family/general practice]:age...,2.498977,-2.498977,statsmodels,2.0,0.000290,-0.000290,65.0,0.001402,0.000395,44.0
2,specialty_group[T.Other]:age_group3[T.30 years...,2.409895,-2.409895,statsmodels,3.0,0.001744,-0.001744,36.0,0.009578,0.000910,8.0
3,specialty_group[T.Internal Medicine]:age_group...,2.023971,-2.023971,statsmodels,4.0,0.000509,-0.000509,58.0,0.002192,0.000544,31.0
4,age_group3[T.30 years old or younger],1.857647,1.857647,statsmodels,5.0,0.004185,0.004185,18.0,0.008207,0.001121,9.0
5,specialty_group[T.Missing or unknown]:age_grou...,1.471311,-1.471311,statsmodels,6.0,0.001459,-0.001459,40.0,0.004044,0.001098,22.0
6,primary_diag_group[T.Musculoskeletal]:hba1c_gr...,0.802534,0.802534,statsmodels,7.0,0.000048,0.000048,83.0,0.000222,0.000106,63.0
7,primary_diag_group[T.Musculoskeletal]:hba1c_gr...,0.796584,0.796584,statsmodels,8.0,0.000103,0.000103,78.0,0.000330,0.000174,56.0
8,primary_diag_group[T.Musculoskeletal],0.726567,-0.726567,statsmodels,9.0,0.006206,-0.000157,11.0,0.010799,0.001289,6.0
9,primary_diag_group[T.Neoplasms]:hba1c_group[T....,0.722161,0.722161,statsmodels,10.0,0.000027,0.000027,86.0,0.000158,0.000083,67.0


#### Representative cases

In [30]:
cases = local_results["cases"]

print("\nRepresentative cases:")

if cases is not None and not cases.empty:

    columns_to_show = [
        col for col in [
            "CaseType",
            "y_true",
            "y_pred",
            "y_prob"
        ]
        if col in cases.columns
    ]

    display(cases[columns_to_show])

else:
    print("No representative cases returned.")


Representative cases:


,CaseType,y_true,y_pred,y_prob
48396,TN,0.0,0,0.009532
48249,FN,1.0,0,0.020809
41626,Borderline,0.0,0,0.352826


#### LIME

In [31]:
lime_df = local_results["lime"]

print("\nLIME rows:", len(lime_df))

if not lime_df.empty:
    display(lime_df.head(20))
else:
    print("WARNING: LIME output is empty.")


LIME rows: 30


,CaseIndex,Prediction,Probability,Rule,Weight
0,48396,0,0.009532,specialty_group[T.Other]:age_group3[T.30 years...,0.464301
1,48396,0,0.009532,specialty_group[T.Surgery]:age_group3[T.30 yea...,-0.438146
2,48396,0,0.009532,specialty_group[T.Family/general practice]:age...,0.419764
3,48396,0,0.009532,specialty_group[T.Internal Medicine]:age_group...,0.409263
4,48396,0,0.009532,specialty_group[T.Missing or unknown]:age_grou...,0.305060
5,48396,0,0.009532,age_group3[T.30 years old or younger] > 0.00,0.244564
6,48396,0,0.009532,time_in_hospital > 6.00,0.152329
7,48396,0,0.009532,primary_diag_group[T.Musculoskeletal] <= 0.00,0.146753
8,48396,0,0.009532,primary_diag_group[T.Injury]:hba1c_group[T.Nor...,0.097986
9,48396,0,0.009532,primary_diag_group[T.Neoplasms]:hba1c_group[T....,-0.064703


#### Stability

In [32]:
if stability_results is None:
    print(
        "\nStability validation: N/A "
        "(expected for HbA1c full-sample statsmodels model)"
    )
else:
    display(stability_results["raw"].head())
    print(stability_results["summary"])



Stability validation: N/A (expected for HbA1c full-sample statsmodels model)


#### Exported folder

In [33]:
out_dir = Path(config["output_dir"]).resolve()

print("\nOutput directory:")
print(out_dir)

print("\nExported files:")

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))


print("\nHBA1C XAI FRAMEWORK CHECKS COMPLETE (IN-SAMPLE)")


Output directory:
C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\hba1c

Exported files:
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
lime_explanations.csv
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv

HBA1C XAI FRAMEWORK CHECKS COMPLETE (IN-SAMPLE)


In [34]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)

['Intercept', 'gender[T.Male]', 'race_group[T.Caucasian]', 'race_group[T.Missing]', 'race_group[T.Other]', 'admission_source_group[T.Admitted because of physician/clinic referral]', 'admission_source_group[T.Otherwise]', 'discharge_group[T.Otherwise]', 'specialty_group[T.Family/general practice]', 'specialty_group[T.Internal Medicine]', 'specialty_group[T.Missing or unknown]', 'specialty_group[T.Other]', 'specialty_group[T.Surgery]', 'primary_diag_group[T.Circulatory]', 'primary_diag_group[T.Digestive]', 'primary_diag_group[T.Genitourinary]', 'primary_diag_group[T.Injury]', 'primary_diag_group[T.Musculoskeletal]', 'primary_diag_group[T.Neoplasms]', 'primary_diag_group[T.Other]', 'primary_diag_group[T.Respiratory]', 'age_group3[T.Older than 60]', 'age_group3[T.30 years old or younger]', 'hba1c_group[T.Normal result of the test]', 'hba1c_group[T.Result was high and the diabetic medication was changed]', 'hba1c_group[T.Result was high but the diabetic medication was not changed]', 'discha

,Intercept,gender[T.Male],race_group[T.Caucasian],race_group[T.Missing],race_group[T.Other],admission_source_group[T.Admitted because of physician/clinic referral],admission_source_group[T.Otherwise],discharge_group[T.Otherwise],specialty_group[T.Family/general practice],specialty_group[T.Internal Medicine],...,primary_diag_group[T.Musculoskeletal]:time_in_hospital,primary_diag_group[T.Neoplasms]:time_in_hospital,primary_diag_group[T.Other]:time_in_hospital,primary_diag_group[T.Respiratory]:time_in_hospital,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
48396,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0.009532,0.490468,TN,48396
48249,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,1.0,0,0.020809,0.479191,FN,48249
41626,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0,0.352826,0.147174,Borderline,41626


In [35]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns
print("HbA1c CaseIndex exists in memory.")

HbA1c CaseIndex exists in memory.


In [36]:
hba1c_cases = pd.read_csv(
    Path("../../results/hba1c/representative_cases.csv")
)

print(hba1c_cases.columns.tolist())
display(hba1c_cases)

['Intercept', 'gender[T.Male]', 'race_group[T.Caucasian]', 'race_group[T.Missing]', 'race_group[T.Other]', 'admission_source_group[T.Admitted because of physician/clinic referral]', 'admission_source_group[T.Otherwise]', 'discharge_group[T.Otherwise]', 'specialty_group[T.Family/general practice]', 'specialty_group[T.Internal Medicine]', 'specialty_group[T.Missing or unknown]', 'specialty_group[T.Other]', 'specialty_group[T.Surgery]', 'primary_diag_group[T.Circulatory]', 'primary_diag_group[T.Digestive]', 'primary_diag_group[T.Genitourinary]', 'primary_diag_group[T.Injury]', 'primary_diag_group[T.Musculoskeletal]', 'primary_diag_group[T.Neoplasms]', 'primary_diag_group[T.Other]', 'primary_diag_group[T.Respiratory]', 'age_group3[T.Older than 60]', 'age_group3[T.30 years old or younger]', 'hba1c_group[T.Normal result of the test]', 'hba1c_group[T.Result was high and the diabetic medication was changed]', 'hba1c_group[T.Result was high but the diabetic medication was not changed]', 'discha

,Intercept,gender[T.Male],race_group[T.Caucasian],race_group[T.Missing],race_group[T.Other],admission_source_group[T.Admitted because of physician/clinic referral],admission_source_group[T.Otherwise],discharge_group[T.Otherwise],specialty_group[T.Family/general practice],specialty_group[T.Internal Medicine],...,primary_diag_group[T.Musculoskeletal]:time_in_hospital,primary_diag_group[T.Neoplasms]:time_in_hospital,primary_diag_group[T.Other]:time_in_hospital,primary_diag_group[T.Respiratory]:time_in_hospital,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,0.009532,0.490468,TN,48396
1,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,1.0,0,0.020809,0.479191,FN,48249
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0,0.352826,0.147174,Borderline,41626
